# DriftCheck outputs with Qwen Image Edit

Makes 3 AI outputs per product with the DriftCheck prompts.

Status: written for Kaggle but not yet run there. Run the smoke test first.

Steps:
1. On your laptop run `driftcheck jobs --generator qwen_edit`. It writes `data\jobs\qwen_edit\kaggle_input.zip`.
2. On Kaggle, create a new dataset and upload that zip. Kaggle unpacks it.
3. Open this notebook on Kaggle. Add the dataset with Add Input. Settings: Accelerator GPU T4 x2 or P100, Internet on.
4. Run all cells with `SMOKE_TEST_ONLY = True`. Look at the one test image.
5. If it looks right, set `SMOKE_TEST_ONLY = False` and run again. Done jobs are skipped, so a rerun picks up where it stopped.
6. Download `qwen_edit_outputs.zip` from the Output tab.
7. On your laptop:

```
Expand-Archive qwen_edit_outputs.zip -DestinationPath data\jobs\qwen_edit\result
.venv\Scripts\driftcheck register --from data\jobs\qwen_edit\result
```

In [ ]:
# Settings
SMOKE_TEST_ONLY = True

# 4 bit community build of Qwen-Image-Edit-2511, so it fits a 16 GB GPU with CPU offload.
# Base model: Qwen/Qwen-Image-Edit-2511. Change MODEL_ID if this repo is gone.
MODEL_ID = "ovedrive/Qwen-Image-Edit-2511-4bit"
GENERATOR = "qwen_edit"
STEPS = 20
TRUE_CFG_SCALE = 4.0
SEED = 0
MAX_SIDE = 1024

# Optional 4 step Lightning LoRA. Faster, but not confirmed to work on the 4 bit build.
USE_LIGHTNING = False
LIGHTNING_REPO = "lightx2v/Qwen-Image-Edit-2511-Lightning"
LIGHTNING_WEIGHTS = "Qwen-Image-Edit-2511-Lightning-4steps-V1.0-bf16.safetensors"
if USE_LIGHTNING:
    STEPS, TRUE_CFG_SCALE = 4, 1.0

In [ ]:
!pip install -q -U diffusers transformers accelerate bitsandbytes peft

In [ ]:
import json, os, shutil, time, zipfile
from datetime import date
from pathlib import Path

import numpy as np
import torch
from PIL import Image, ImageOps

!nvidia-smi --query-gpu=name,memory.total --format=csv
assert torch.cuda.is_available(), "No GPU. Set Accelerator to GPU in the notebook settings."
DTYPE = torch.bfloat16
print("bf16 native:", torch.cuda.is_bf16_supported(including_emulation=False))

found = sorted(Path("/kaggle/input").glob("**/jobs.json"))
assert found, "jobs.json not found. Add the kaggle_input dataset with Add Input."
INPUT_DIR = found[0].parent
OUT_DIR = Path("/kaggle/working/qwen_edit_outputs")
JOBS = json.loads((INPUT_DIR / "jobs.json").read_text())
print(len(JOBS), "jobs from", INPUT_DIR)

In [ ]:
from diffusers import QwenImageEditPlusPipeline

pipe = QwenImageEditPlusPipeline.from_pretrained(MODEL_ID, torch_dtype=DTYPE)
if USE_LIGHTNING:
    pipe.load_lora_weights(LIGHTNING_REPO, weight_name=LIGHTNING_WEIGHTS)
pipe.enable_model_cpu_offload()
pipe.set_progress_bar_config(disable=True)
print("pipeline ready")

In [ ]:
def out_paths(job):
    folder = OUT_DIR / "products" / job["product_id"] / "outputs"
    return folder / f"{job['output_id']}.jpg", folder / f"{job['output_id']}.json"


def load_original(product_id):
    img = Image.open(INPUT_DIR / "products" / product_id / "original.jpg")
    img = ImageOps.exif_transpose(img).convert("RGB")
    img.thumbnail((MAX_SIDE, MAX_SIDE), Image.Resampling.LANCZOS)
    return img


def run_job(job):
    image_path, json_path = out_paths(job)
    if image_path.exists():
        return None
    start = time.perf_counter()
    with torch.inference_mode():
        result = pipe(
            image=[load_original(job["product_id"])],
            prompt=job["prompt"],
            negative_prompt=" ",
            true_cfg_scale=TRUE_CFG_SCALE,
            num_inference_steps=STEPS,
            generator=torch.manual_seed(SEED),
        ).images[0]
    seconds = time.perf_counter() - start
    # A black image means the numbers overflowed. Do not save it as an output.
    if np.asarray(result).mean() < 3:
        raise RuntimeError("Output is black. The GPU may not handle bf16 for this model. Try the P100 or another GPU.")
    image_path.parent.mkdir(parents=True, exist_ok=True)
    result.convert("RGB").save(image_path, quality=95)
    record = {
        "product_id": job["product_id"],
        "output_id": job["output_id"],
        "style": job["style"],
        "prompt": job["prompt"],
        "generator": GENERATOR,
        "model": MODEL_ID + (" + " + LIGHTNING_REPO if USE_LIGHTNING else ""),
        "date": date.today().isoformat(),
        "steps": STEPS,
        "seed": SEED,
        "seconds": round(seconds, 1),
    }
    json_path.write_text(json.dumps(record, indent=2, ensure_ascii=False))
    return seconds


def zip_outputs():
    archive = Path("/kaggle/working/qwen_edit_outputs.zip")
    with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
        for path in OUT_DIR.rglob("*"):
            if path.is_file():
                z.write(path, path.relative_to(OUT_DIR))
    return archive

In [ ]:
# Smoke test on the first job
first = JOBS[0]
seconds = run_job(first)
print(first["output_id"], "for", first["product_id"], "took", seconds, "seconds (None means it already existed)")
display(load_original(first["product_id"]).resize((384, 384)))
display(Image.open(out_paths(first)[0]).resize((384, 384)))

In [ ]:
# Full run. Each job is saved as soon as it finishes, and the zip is refreshed every 10 jobs.
if SMOKE_TEST_ONLY:
    print("Smoke test only. Set SMOKE_TEST_ONLY = False and run again for all jobs.")
else:
    failed = []
    for i, job in enumerate(JOBS, 1):
        try:
            seconds = run_job(job)
            if seconds is not None:
                print(f"{i}/{len(JOBS)} {job['product_id']} {job['output_id']} {seconds:.0f}s")
        except Exception as error:
            failed.append((job["product_id"], job["output_id"], str(error)))
            print(f"{i}/{len(JOBS)} {job['product_id']} {job['output_id']} FAILED: {error}")
        if i % 10 == 0:
            zip_outputs()
    print("failed:", failed)

print("zip:", zip_outputs())